# 🛍️ Customer Segmentation Using RFM Analysis
### E-Commerce Behavioral Analytics | UCI Online Retail Dataset

**Objective:** Segment customers based on purchasing behavior to identify 
high-value customers, at-risk churners, and growth opportunities.

**Tools:** Python · pandas · matplotlib · seaborn · plotly

**Dataset:** UCI Online Retail Dataset — 541,909 transactions from a UK-based 
online retailer (2010–2011)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries loaded successfully")

In [ ]:
# Load dataset
df = pd.read_excel('C:\\Project\\customer-segmentation-rfm\\data\\Online Retail.xlsx', engine='openpyxl')

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 1. Exploratory Data Analysis (EDA)

Before cleaning, let's understand what we're working with.

In [ ]:
print("=== BASIC INFO ===")
print(f"Total rows     : {df.shape[0]:,}")
print(f"Total columns  : {df.shape[1]}")
print(f"Date range     : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Unique products : {df['StockCode'].nunique():,}")
print(f"Countries       : {df['Country'].nunique()}")

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

## 2. Data Cleaning

Issues to address:
- Remove rows with missing `CustomerID` (can't do RFM without it)
- Remove cancelled orders (InvoiceNo starting with 'C')
- Remove rows where Quantity or UnitPrice ≤ 0
- Keep only United Kingdom for focused analysis

In [ ]:
df_clean = df.copy()

# Drop missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])

# Remove cancellations
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative/zero quantity and price
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Focus on UK customers
df_clean = df_clean[df_clean['Country'] == 'United Kingdom']

# Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Ensure correct date type
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Rows before cleaning : {df.shape[0]:,}")
print(f"Rows after cleaning  : {df_clean.shape[0]:,}")
print(f"Rows removed         : {df.shape[0] - df_clean.shape[0]:,}")
print(f"\nCustomers remaining  : {df_clean['CustomerID'].nunique():,}")

## 3. RFM Calculation

- **Recency (R):** How recently did the customer make a purchase?
- **Frequency (F):** How often do they purchase?
- **Monetary (M):** How much total revenue do they generate?

In [ ]:
# Snapshot date = 1 day after last transaction
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

# Calculate RFM
rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f"\nRFM Table Shape: {rfm.shape}")
rfm.describe()

In [ ]:
# Score each metric 1-5 (5 = best)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  q=5, labels=[1,2,3,4,5])

# Combine into RFM Score string
rfm['RFM_Score'] = (rfm['R_Score'].astype(str) + 
                    rfm['F_Score'].astype(str) + 
                    rfm['M_Score'].astype(str))

# Total RFM Score (numeric)
rfm['RFM_Total'] = (rfm['R_Score'].astype(int) + 
                    rfm['F_Score'].astype(int) + 
                    rfm['M_Score'].astype(int))

rfm.head(10)

In [ ]:
def segment_customer(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r >= 3 and f <= 2 and m >= 3:
        return 'Potential Loyalists'
    elif r == 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Can\'t Lose Them'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost / Inactive'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Segment summary
segment_summary = rfm.groupby('Segment').agg(
    Customer_Count = ('CustomerID', 'count'),
    Avg_Recency    = ('Recency',    'mean'),
    Avg_Frequency  = ('Frequency',  'mean'),
    Avg_Monetary   = ('Monetary',   'mean'),
    Total_Revenue  = ('Monetary',   'sum')
).round(2).sort_values('Total_Revenue', ascending=False)

print(segment_summary)